# Online Retail II: cohort retention and RFM segmentation

A walkthrough of the analysis. All of the logic lives in the SQL files under `sql/`. This notebook runs them with DuckDB and looks at the results.

**Business questions**
1. How healthy is the raw data, and what has to be cleaned?
2. How does revenue move over time? Is there seasonality?
3. Do new customers come back? (cohort retention)
4. Which customers matter most, and who is about to churn? (RFM)
5. How concentrated is revenue across customers, countries and products?

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from pipeline import build

pd.set_option('display.width', 140)
con = build()
sql = lambda q: con.sql(q).df()

## 1. Data quality
Measured on the raw file before any cleaning (`sql/01_data_quality.sql`).

In [2]:
sql('SELECT * FROM data_quality')

,check_name,rows_affected,pct_of_rows
0,Exact duplicate rows,34335,3.22
1,Missing customer ID,243007,22.77
2,"Cancellation lines (invoice starts with ""C"")",19494,1.83
3,"Bad-debt adjustments (invoice starts with ""A"")",6,0.00
4,"Non-product stock codes (postage, fees, manual...",6093,0.57
5,Zero or negative quantity,22950,2.15
6,Zero or negative unit price,6207,0.58
7,"Single lines with 10,000+ units (possible entr...",15,0.00


About 23% of lines have no customer ID. They are kept for revenue totals but left out of customer-level analysis (cohorts, RFM). The 15 lines with 10,000+ units led to an extra rule: orders that were cancelled in full on the same day are entry mistakes, so both sides are removed.

In [3]:
sql('''SELECT count(*) AS reversed_lines,
              round(sum(reversed_value)) AS reversed_value_gbp
       FROM same_day_reversals''')

,reversed_lines,reversed_value_gbp
0,1382,419724.0


In [4]:
sql('''SELECT stock_code, quantity, unit_price, sale_time, cancel_time
       FROM same_day_reversals ORDER BY reversed_value DESC LIMIT 5''')

,stock_code,quantity,unit_price,sale_time,cancel_time
0,23843,80995,2.08,2011-12-09 09:15:00,2011-12-09 09:27:00
1,23166,74215,1.04,2011-01-18 10:01:00,2011-01-18 10:17:00
2,M,1,10953.50,2010-03-23 15:22:00,2010-03-23 15:24:00
3,M,1,8985.60,2010-01-29 11:04:00,2010-01-29 11:05:00
4,M,1,6958.17,2010-03-19 11:35:00,2010-03-19 11:49:00


## 2. Clean dataset

In [5]:
sql('SELECT * FROM headline_kpis')

,first_day,last_day,sales_lines,orders,customers,products,revenue,avg_order_value
0,2009-12-01,2011-12-09,1001936,39397,5850,4873,19294885.0,489.76


## 3. Revenue over time

In [6]:
sql('SELECT * FROM monthly_kpis')

,order_month,revenue,orders,active_customers,avg_order_value,revenue_mom_pct,revenue_yoy_pct
0,2009-12-01,795053.25,1661,951,478.66,NaN,NaN
1,2010-01-01,611941.62,1047,700,584.47,-23.0,NaN
2,2010-02-01,537426.39,1184,769,453.91,-12.2,NaN
3,2010-03-01,760577.94,1644,1050,462.64,41.5,NaN
4,2010-04-01,643812.51,1431,938,449.90,-15.4,NaN
5,2010-05-01,629019.65,1474,962,426.74,-2.3,NaN
6,2010-06-01,682884.43,1608,1033,424.68,8.6,NaN
7,2010-07-01,625161.70,1504,924,415.67,-8.5,NaN
8,2010-08-01,673139.39,1399,909,481.16,7.7,NaN
9,2010-09-01,864187.22,1785,1134,484.14,28.4,NaN


Revenue roughly triples between the quietest months (Feb/Apr) and November. Nov 2011 is only 1.4% above Nov 2010, so year-on-year growth is flat at the peak.

## 4. Cohort retention

In [7]:
sql('SELECT * FROM retention_summary')

,month_number,avg_retention_pct,cohorts_included
0,1,20.9,22
1,3,21.1,20
2,6,18.5,17
3,12,18.4,11


In [8]:
heat = sql('SELECT * FROM cohort_retention WHERE month_number BETWEEN 1 AND 12')
heat.pivot(index='cohort_month', columns='month_number', values='retention_pct')

month_number,1,2,3,4,5,6,7,8,9,10,11,12
cohort_month,,,,,,,,,,,,
2009-12-01,34.9,33.3,42.5,37.9,36.0,37.6,34.4,33.6,36.1,42.2,49.6,37.6
2010-01-01,21.2,31.8,31.5,26.9,31.0,26.9,23.4,28.5,32.6,30.7,17.9,22.8
2010-02-01,23.5,22.7,29.4,24.6,19.5,19.3,28.9,25.7,27.8,11.5,12.6,15.0
2010-03-01,19.0,23.1,24.3,23.1,20.4,24.7,30.6,27.7,10.9,11.3,14.5,20.0
2010-04-01,19.1,19.1,16.0,18.4,22.2,27.6,26.6,10.6,10.9,7.5,14.0,14.3
2010-05-01,15.4,17.0,17.8,17.8,25.7,21.3,12.6,5.9,8.3,11.5,13.4,15.4
2010-06-01,17.6,18.7,20.6,23.2,28.5,12.7,9.0,8.2,11.2,10.5,13.9,15.0
2010-07-01,15.6,18.3,29.6,29.0,14.0,11.3,15.1,14.5,11.3,13.4,14.5,12.9
2010-08-01,19.6,28.8,32.5,16.6,11.7,9.8,12.9,13.5,12.9,12.9,11.7,14.7


Average retention is roughly flat at about 20% from month 1 to month 3, then about 18% to month 12. Customers who stay tend to keep buying, but most new customers never come back in a given month. The autumn 2010 cohorts jump at month 12, so seasonal gift buyers return for the next Christmas.

In [9]:
sql('SELECT * FROM repeat_rate')

,customers,repeat_customers,repeat_rate_pct
0,5850,4230,72.3


## 5. RFM segmentation

In [10]:
sql('SELECT * FROM rfm_segments')

,segment,customers,pct_customers,revenue,pct_revenue,avg_recency_days,avg_orders,avg_revenue_per_customer
0,Champions,1464,25.0,11694260.0,69.9,21.0,15.5,7988.0
1,Loyal,1216,20.8,2420901.0,14.5,78.0,5.4,1991.0
2,At Risk,830,14.2,1493983.0,8.9,368.0,5.0,1800.0
3,Lost,1270,21.7,318253.0,1.9,464.0,1.2,251.0
4,Hibernating (high value),240,4.1,305570.0,1.8,417.0,1.6,1273.0
5,Needs Attention,385,6.6,245281.0,1.5,106.0,1.4,637.0
6,New / Promising,445,7.6,242513.0,1.5,28.0,1.5,545.0


In [11]:
sql('''SELECT segment, r_score, f_score, count(*) AS customers
       FROM rfm GROUP BY ALL ORDER BY r_score DESC, f_score DESC LIMIT 10''')

,segment,r_score,f_score,customers
0,Champions,5,5,564
1,Champions,5,4,261
2,Loyal,5,3,184
3,New / Promising,5,2,109
4,New / Promising,5,1,52
5,Champions,4,5,335
6,Champions,4,4,304
7,Loyal,4,3,247
8,New / Promising,4,2,169
9,New / Promising,4,1,115


## 6. Concentration: customers, countries, products

In [12]:
sql('SELECT * FROM customer_pareto WHERE pct_customers IN (1, 5, 10, 20, 50)')

,customer_rank,pct_customers,cumulative_pct_revenue
0,585,10.0,63.29
1,1170,20.0,76.78
2,2925,50.0,93.42


In [13]:
sql('SELECT * FROM country_revenue LIMIT 8')

,country,revenue,orders,customers,pct_revenue
0,United Kingdom,16463519.0,36087,5333,85.33
1,EIRE,615156.0,579,3,3.19
2,Netherlands,549773.0,216,22,2.85
3,Germany,383169.0,753,107,1.99
4,France,309853.0,595,93,1.61
5,Australia,167423.0,85,15,0.87
6,Spain,97767.0,144,38,0.51
7,Switzerland,93634.0,83,22,0.49


In [14]:
sql('SELECT * FROM product_performance LIMIT 10')

,stock_code,description,units_sold,revenue,orders,units_cancelled,cancel_rate_pct
0,22423,REGENCY CAKESTAND 3 TIER,26291.0,328472.0,3904,1263.0,4.8
1,85123A,CREAM HANGING HEART T-LIGHT HOLDER,93775.0,256629.0,5362,3209.0,3.4
2,85099B,JUMBO BAG RED RETROSPOT,95991.0,179209.0,3979,547.0,0.6
3,47566,PARTY BUNTING,28190.0,148271.0,2672,275.0,1.0
4,84879,ASSORTED COLOUR BIRD ORNAMENT,79869.0,129003.0,2803,298.0,0.4
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,34716.0,116819.0,2010,263.0,0.8
6,79321,CHILLI LIGHTS,15788.0,80309.0,1131,116.0,0.7
7,22197,POPCORN HOLDER,88299.0,79376.0,2321,604.0,0.7
8,22386,JUMBO BAG PINK POLKADOT,38927.0,75420.0,2240,72.0,0.2
9,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,31409.0,71300.0,756,9381.0,29.9


In [15]:
sql('''SELECT * FROM product_performance
       WHERE cancel_rate_pct IS NOT NULL
       ORDER BY cancel_rate_pct DESC LIMIT 5''')

,stock_code,description,units_sold,revenue,orders,units_cancelled,cancel_rate_pct
0,23113,PANTRY CHOPPING BOARD,1154.0,5869.0,57,946.0,82.0
1,79341,WILLOW BRANCH LIGHTS.,725.0,4424.0,79,448.0,61.8
2,23057,BEADED CHANDELIER T-LIGHT HOLDER,581.0,1481.0,43,337.0,58.0
3,84493,BLUE/PINK NEST STRIPE BOX,1142.0,111.0,5,567.0,49.6
4,84924F,PSYCHEDELIC METAL SIGN CALENDAR,975.0,275.0,10,480.0,49.2


In [16]:
sql('SELECT * FROM cancellation_summary')

,gross_sales,cancelled_value,cancelled_pct_of_sales
0,19294885.0,377376.0,2.0


## 7. Recommendations
- **Win back "At Risk" customers** (frequent buyers who have gone quiet). They are about 14% of customers and almost £1.5M of past revenue.
- **Protect the Champions.** A quarter of customers bring in about 70% of revenue, so account management and early access to stock for the Q4 rush are worth it.
- **Plan stock and staffing for Sept–Nov**, when revenue nearly triples.
- **Look into high-return products** (e.g. the pantry chopping board at 82% returned) before reordering them.
- **Collect customer IDs on every order.** About 13% of revenue cannot be linked to a customer.

**Limitations:** single retailer, mostly wholesale buyers, 2 years of data, and no cost data, so this looks at revenue only, not profit.